In [1]:
# ============================================================
# Free scalar propagator on 4D periodic lattice with JAX (GPU)
# Multi-L test of m_eff(L) and Laplacian gap scaling
# with analytic cross-check for C(t)
# ============================================================
import os
import time
import numpy as np
import jax
import jax.numpy as jnp
from jax import lax
from functools import partial

# JAX config: 64-bit, GPU if available
jax.config.update("jax_enable_x64", True)
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["XLA_PYTHON_CLIENT_ALLOCATOR"] = "platform"

print("JAX backend:", jax.default_backend())

# ------------------------------------------------------------
# 1. 4D periodic Laplacian Δ and operator H = m0^2 - Δ
# ------------------------------------------------------------
def laplacian_4d(phi):
    """
    4D periodic scalar Laplacian:
      (Δ φ)(x) = Σ_μ [ φ(x+μ) + φ(x-μ) - 2 φ(x) ]
    with μ = 0,1,2,3 corresponding to the four axes.
    """
    res = jnp.zeros_like(phi)
    for mu in range(4):
        res = res + jnp.roll(phi, +1, axis=mu) + jnp.roll(phi, -1, axis=mu) - 2.0 * phi
    return res

@jax.jit
def apply_operator(phi, m0):
    """
    Operator H = m0^2 - Δ  (positive definite for m0>0)
    NOTE: m0 is NOT static here (fixes the non-hashable static arg issue).
    """
    return m0**2 * phi - laplacian_4d(phi)


# ------------------------------------------------------------
# 2. Conjugate Gradient solver for (m0^2 - Δ) φ = b
# ------------------------------------------------------------
@partial(jax.jit, static_argnums=(2,))
def cg_solve(b, m0, maxiter=300):
    """
    Conjugate Gradient solve of H φ = b, with H = m0^2 - Δ.
    Returns (phi, history of ||r||^2).

    maxiter is static so that lax.scan has a concrete length.
    """
    x0 = jnp.zeros_like(b)
    r0 = b - apply_operator(x0, m0)
    p0 = r0
    r2_0 = jnp.vdot(r0, r0).real

    def body(carry, i):
        x, r, p, r2 = carry
        Ap = apply_operator(p, m0)
        alpha = r2 / (jnp.vdot(p, Ap).real + 1e-16)
        x_new = x + alpha * p
        r_new = r - alpha * Ap
        r2_new = jnp.vdot(r_new, r_new).real
        beta = r2_new / (r2 + 1e-16)
        p_new = r_new + beta * p
        return (x_new, r_new, p_new, r2_new), r2_new

    (x_fin, r_fin, p_fin, r2_fin), r2_hist = lax.scan(
        body,
        (x0, r0, p0, r2_0),
        jnp.arange(maxiter)
    )
    return x_fin, r2_hist


# ------------------------------------------------------------
# 3. Zero-momentum correlator and effective mass (cosh)
# ------------------------------------------------------------
def zero_momentum_correlator(phi):
    """
    Given solution φ(t,x,y,z) to (m0^2 - Δ) φ = δ_(t=0,x=0,y=0,z=0),
    build the zero-momentum two-point:
      C(t) = Σ_{x,y,z} φ(t,x,y,z)
    This is the free 2-pt function at zero spatial momentum.
    """
    C_t = jnp.sum(phi, axis=(1, 2, 3))
    return C_t

def effective_mass_cosh(C):
    """
    Cosh-based effective mass for periodic correlator:
      m_eff(t) = arccosh([C(t-1)+C(t+1)] / [2 C(t)])
    for 1 <= t <= T-2.

    For the free scalar zero-momentum correlator, this should be
    essentially constant and equal to the lattice energy E_lat(m0).
    """
    C_np = np.array(C, dtype=float)
    T = len(C_np)
    meff = np.full(T, np.nan, dtype=float)

    for t in range(1, T-1):
        den = 2.0 * C_np[t]
        if den <= 0:
            continue
        num = C_np[t-1] + C_np[t+1]
        x = num / den
        if x < 1.0:
            # numerical noise can push slightly below 1
            continue
        meff[t] = np.arccosh(x)
    return meff


# ------------------------------------------------------------
# 4. Analytic zero-momentum correlator on L^4
# ------------------------------------------------------------
def analytic_zero_momentum_C(L, m0):
    """
    Analytic C(t) for free scalar on L^4 with periodic BC, zero spatial momentum:

      C(t) = (1/L) Σ_{n=0}^{L-1} cos(p0 t) / (m0^2 + 4 sin^2(p0/2)),
      p0 = 2π n / L.

    This satisfies the same (m0^2 - Δ_1D) equation and gives
    a correlator with exact single-particle energy

      cosh(E_lat) = 1 + m0^2 / 2.

    We use this as a reference to check the CG-based solution.
    """
    C = np.zeros(L, dtype=np.float64)
    for t in range(L):
        s = 0.0
        for n0 in range(L):
            p0 = 2.0 * np.pi * n0 / L
            hatp2 = 4.0 * np.sin(p0 / 2.0)**2
            s += np.cos(p0 * t) / (m0**2 + hatp2)
        C[t] = s / L
    return C


# ------------------------------------------------------------
# 5. Single-L test: δ-source, solve, compare to analytic, m_eff
# ------------------------------------------------------------
def run_single_L(L, m0=0.5, maxiter=300, plateau_range=(4, 12)):
    """
    Run the free scalar propagator test on a 4D L^4 lattice.

    Returns:
      C_num, meff_num, plateau_m, E_lat, lam_min, L2_lam_min
    """
    print(f"\n=== FREE SCALAR PROPAGATOR TEST (L={L}, m0={m0}) ===")

    shape = (L, L, L, L)

    # Point source δ(t=0,x=0,y=0,z=0)
    b = jnp.zeros(shape, dtype=jnp.float64)
    b = b.at[0, 0, 0, 0].set(1.0)

    print(f"Solving (m0^2 - Δ) φ = δ with CG (maxiter={maxiter}) ...")

    # Warmup compile
    _ = cg_solve(b, m0, maxiter=2)

    t0 = time.time()
    phi, r2_hist = cg_solve(b, m0, maxiter=maxiter)
    jax.block_until_ready(phi)
    dt = time.time() - t0

    r2_np = np.array(r2_hist)
    r_np = np.sqrt(r2_np + 0.0)

    print(f"[CG] initial ||r|| = {r_np[0]: .4e}")
    for k in range(0, maxiter, 20):
        print(f"[CG] iter {k:4d}, ||r|| = {r_np[k]: .4e}")
    print(f"[CG] iter {maxiter-1:4d}, ||r|| = {r_np[maxiter-1]: .4e}")
    print(f"[CG] finished in {dt:.2f} s, final ||r|| = {r_np[maxiter-1]: .4e}")

    print("\nComputing zero-momentum C_num(t) and m_eff(t) ...")

    # Numerical zero-momentum correlator from JAX solution
    C_num = zero_momentum_correlator(phi)
    C_num_np = np.array(C_num, dtype=float)
    meff_num = effective_mass_cosh(C_num)
    meff_num_np = np.array(meff_num, dtype=float)

    # Analytic reference correlator and effective mass
    C_ref_np = analytic_zero_momentum_C(L, m0)
    meff_ref_np = effective_mass_cosh(C_ref_np)

    # L∞ error between numerical and analytic correlators
    max_abs_diff = np.max(np.abs(C_num_np - C_ref_np))

    print("\n=== ZERO-MOMENTUM CORRELATOR C(t) AND m_eff(t) (NUMERIC) ===")
    print("  t |      C_num(t) |   m_eff_num(t)")
    print("--------------------------------------")
    for t in range(L):
        Ct = C_num_np[t]
        mt = meff_num_np[t]
        if np.isnan(mt):
            print(f"{t:3d} | {Ct: .4e} |")
        else:
            print(f"{t:3d} | {Ct: .4e} | {mt:10.4f}")

    print("\n=== ANALYTIC ZERO-MOMENTUM C_ref(t) AND m_eff_ref(t) ===")
    print("  t |      C_ref(t) |   m_eff_ref(t)")
    print("--------------------------------------")
    for t in range(L):
        Ct = C_ref_np[t]
        mt = meff_ref_np[t]
        if np.isnan(mt):
            print(f"{t:3d} | {Ct: .4e} |")
        else:
            print(f"{t:3d} | {Ct: .4e} | {mt:10.4f}")

    print(f"\nMax |C_num(t) - C_ref(t)| over t = {max_abs_diff:.3e}")

    # Plateau estimate for numerical m_eff over given window
    t0_pl, t1_pl = plateau_range
    idx = np.arange(L)
    mask = (~np.isnan(meff_num_np)) & (idx >= t0_pl) & (idx < t1_pl)
    plateau_m = np.mean(meff_num_np[mask]) if np.any(mask) else np.nan

    # Theoretical single-particle lattice energy:
    # cosh(E_lat) = 1 + m0^2 / 2
    E_lat = float(np.arccosh(1.0 + 0.5 * m0**2))

    # Smallest non-zero eigenvalue of -Δ in the time direction:
    lam_min = 4.0 * np.sin(np.pi / L)**2
    L2_lam_min = (L**2) * lam_min

    rel_err = abs(plateau_m - E_lat) / E_lat if not np.isnan(plateau_m) else np.nan

    print("\n=== SUMMARY FOR THIS L ===")
    print(f"L               = {L}")
    print(f"Plateau m_eff   ≈ {plateau_m:.6f}")
    print(f"E_lat(m0)       = {E_lat:.6f}")
    print(f"Relative error  ≈ {100*rel_err:.3f}%")
    print(f"λ_min(-Δ_1D)    = {lam_min:.6f}")
    print(f"L^2 * λ_min     = {L2_lam_min:.6f}")

    return C_num_np, meff_num_np, plateau_m, E_lat, lam_min, L2_lam_min


# ------------------------------------------------------------
# 6. Multi-L scan and scaling summary
# ------------------------------------------------------------
def main():
    m0 = 0.5
    maxiter = 300
    # L values to test
    L_values = [12, 16, 20, 24]

    results = []

    for L in L_values:
        # Plateau window: start a bit away from t=0, end before T/2
        t0_pl = max(3, L // 6)
        t1_pl = L // 2
        C, meff, m_pl, E_lat, lam_min, L2_lam_min = run_single_L(
            L=L, m0=m0, maxiter=maxiter, plateau_range=(t0_pl, t1_pl)
        )
        rel_err = abs(m_pl - E_lat) / E_lat if not np.isnan(m_pl) else np.nan
        results.append((L, m_pl, E_lat, rel_err, lam_min, L2_lam_min))

    print("\n\n=== L-SCALING SUMMARY (FREE SCALAR) ===")
    print("  L |      m_eff |      E_lat |  rel_err% |  λ_min(-Δ_1D) |  L^2 λ_min")
    print("------------------------------------------------------------------------")
    for (L, m_pl, E_lat, rel_err, lam_min, L2_lam_min) in results:
        print(f"{L:3d} | {m_pl:11.6f} | {E_lat:11.6f} | {100*rel_err:9.3f} |"
              f" {lam_min:12.6f} | {L2_lam_min:10.6f}")
    print("------------------------------------------------------------------------")
    print("For a free scalar with mass m0, m_eff(L) should be L-independent.")
    print("Meanwhile λ_min(-Δ_1D) ≈ 4 sin^2(π/L) ∼ 1/L^2 so L^2 λ_min → const (~4π^2).")

if __name__ == "__main__":
    main()

JAX backend: gpu

=== FREE SCALAR PROPAGATOR TEST (L=12, m0=0.5) ===
Solving (m0^2 - Δ) φ = δ with CG (maxiter=300) ...
[CG] initial ||r|| =  3.4284e-01
[CG] iter    0, ||r|| =  3.4284e-01
[CG] iter   20, ||r|| =  1.0889e-03
[CG] iter   40, ||r|| =  3.5119e-08
[CG] iter   60, ||r|| =  1.3089e-09
[CG] iter   80, ||r|| =  9.4075e-10
[CG] iter  100, ||r|| =  7.7495e-10
[CG] iter  120, ||r|| =  6.7486e-10
[CG] iter  140, ||r|| =  6.0594e-10
[CG] iter  160, ||r|| =  5.5473e-10
[CG] iter  180, ||r|| =  5.1472e-10
[CG] iter  200, ||r|| =  4.8233e-10
[CG] iter  220, ||r|| =  4.5542e-10
[CG] iter  240, ||r|| =  4.3258e-10
[CG] iter  260, ||r|| =  4.1289e-10
[CG] iter  280, ||r|| =  3.9568e-10
[CG] iter  299, ||r|| =  3.8118e-10
[CG] finished in 0.32 s, final ||r|| =  3.8118e-10

Computing zero-momentum C_num(t) and m_eff(t) ...

=== ZERO-MOMENTUM CORRELATOR C(t) AND m_eff(t) (NUMERIC) ===
  t |      C_num(t) |   m_eff_num(t)
--------------------------------------
  0 |  9.7527e-01 |
  1 |  5.97

In [2]:
# ============================================================
# SU(3) heat-kernel tensor + HOTRG-like RG flow with Jacobians
# ============================================================
import os
import time
import numpy as np

import jax
import jax.numpy as jnp
from jax import jit, jvp, vjp, random
from functools import partial

import su3_irrep_tensor as su3  # <-- your module

# ---------------------------------
# JAX / GPU configuration
# ---------------------------------
jax.config.update("jax_enable_x64", True)
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["XLA_PYTHON_CLIENT_ALLOCATOR"] = "platform"

print("JAX backend:", jax.default_backend())


# ============================================================
# 1. SU(3) heat-kernel tensor (irrep / right-invariant basis)
# ============================================================

def build_initial_tensor(beta: float, D_cut: int):
    """
    Pull a rank-4 SU(3)-invariant tensor from su3_irrep_tensor.

    Expected shape: (chi, chi, chi, chi), all legs same dim.
    The entries are assumed to already be in the right-invariant
    generator basis (i.e. irrep indices).
    """
    # Try a couple of plausible entry points; tweak to match your module.
    if hasattr(su3, "build_heat_kernel_tensor"):
        T = su3.build_heat_kernel_tensor(beta=beta, D_cut=D_cut)
    elif hasattr(su3, "build_su3_heat_kernel_tensor"):
        T = su3.build_su3_heat_kernel_tensor(beta, D_cut)
    else:
        raise RuntimeError(
            "su3_irrep_tensor must define either "
            "`build_heat_kernel_tensor(beta, D_cut)` or "
            "`build_su3_heat_kernel_tensor(beta, D_cut)`."
        )

    T = jnp.asarray(T, dtype=jnp.complex128)
    if T.ndim != 4:
        raise ValueError(f"Expected rank-4 tensor, got shape {T.shape}")
    chi = T.shape[0]
    if not all(s == chi for s in T.shape):
        raise ValueError(f"Expected all legs same dim, got shape {T.shape}")

    return T  # shape (chi, chi, chi, chi), indices (up, down, left, right)


# ============================================================
# 2. Simple HOTRG-like coarse graining on a rank-4 tensor
# ============================================================

@jit
def hotrg_core(T: jnp.ndarray):
    """
    A simple 2D HOTRG-like map for a uniform rank-4 tensor T[u,d,l,r].

    Scheme:
      1. View T as a matrix M_{(u,l),(d,r)}.
      2. Coarse-grain by squaring the transfer matrix: M' = M @ M.
      3. Reshape back to rank-4, rescale by Frobenius norm to keep
         magnitudes under control.

    This keeps the local bond dimension chi fixed. It's not a full
    textbook HOTRG implementation, but it is:
      - nonlinear in T,
      - translationally invariant,
      - differentiable (so we can build Jacobians).
    """
    chi = T.shape[0]
    # Reorder to (u,l,d,r)
    T_perm = jnp.transpose(T, (0, 2, 1, 3))
    # Flatten (u,l) -> row, (d,r) -> col
    M = T_perm.reshape(chi * chi, chi * chi)
    # Coarse-grain by composing transfer matrix with itself
    M2 = M @ M
    # Normalize to prevent blow-up / underflow
    fro_norm = jnp.linalg.norm(M2)
    M2n = M2 / fro_norm
    # Back to rank-4 tensor, with same index dimensions
    T_perm_new = M2n.reshape(chi, chi, chi, chi)
    T_new = jnp.transpose(T_perm_new, (0, 2, 1, 3))  # (u,d,l,r)
    log_norm = jnp.log(fro_norm)
    return T_new, log_norm


def make_hotrg_ops(chi: int):
    """
    Wrap hotrg_core in a flattened interface, and construct
    matrix-free Jacobian actions J·v and J^T·w on the space
    of tensor entries (right-invariant coordinates).

    theta: flattened T with shape (chi^4,)
    """

    @jit
    def hotrg_step_flat(theta: jnp.ndarray):
        T = theta.reshape((chi, chi, chi, chi))
        T_new, log_norm = hotrg_core(T)
        return T_new.reshape((-1,)), log_norm

    def hotrg_flat_only(theta: jnp.ndarray):
        # Just the tensor map; used for JVP/VJP.
        return hotrg_step_flat(theta)[0]

    def apply_J(theta: jnp.ndarray, v: jnp.ndarray):
        """
        Matrix-free Jacobian action:
          J·v = d/dε [hotrg_flat_only(theta + ε v)]_{ε=0}
        """
        _, Jv = jvp(hotrg_flat_only, (theta,), (v,))
        return Jv

    def apply_JT(theta: jnp.ndarray, w: jnp.ndarray):
        """
        Matrix-free transpose-Jacobian action:
          J^T·w via VJP.
        """
        _, vjp_fun = vjp(hotrg_flat_only, theta)
        (JT_w,) = vjp_fun(w)
        return JT_w

    return hotrg_step_flat, apply_J, apply_JT


# ============================================================
# 3. End-to-end RG run: 20 layers, HOTRG + Jacobians
# ============================================================

def run_rg(beta: float = 5.5, D_cut: int = 64,
           n_layers: int = 20, seed: int = 0):
    """
    End-to-end SU(3) heat-kernel RG run:

      1. Build SU(3) heat-kernel tensor T0 (rank-4, right-invariant basis).
      2. Construct HOTRG-like map hotrg_core and its flattened form.
      3. Construct matrix-free Jacobian actions J·v and J^T·w via JAX.
      4. Run n_layers RG steps, logging:
           - per-layer log_norm (proxy for free energy change),
           - ||T_n||_F,
           - ||J_n·v|| / ||v|| for a random direction v.
    """
    print("\n===============================================")
    print(f" SU(3) HOTRG-LIKE FLOW (beta={beta}, D_cut={D_cut})")
    print("===============================================")

    # 1. Build initial tensor
    print("Building SU(3) heat-kernel tensor...")
    T0 = build_initial_tensor(beta, D_cut)
    chi = T0.shape[0]
    theta0 = T0.reshape((-1,))
    n_params = int(theta0.size)
    print(f"  initial tensor shape: {T0.shape}  (chi={chi})")
    print(f"  number of parameters per layer: {n_params:,}")

    # 2. Build flattened HOTRG map + Jacobian actions
    hotrg_step_flat, apply_J, apply_JT = make_hotrg_ops(chi)

    # Warmup compilation (JIT)
    print("\nJIT-compiling hotrg_step_flat ...")
    t0 = time.time()
    theta_test, logn_test = hotrg_step_flat(theta0)
    theta_test.block_until_ready()
    t1 = time.time()
    print(f"  compilation+first call took {t1 - t0:.2f} s")

    key = random.PRNGKey(seed)
    theta = theta0
    cumulative_log_norm = 0.0

    print("\nRunning RG layers:")
    print("{:>4} | {:>12} | {:>14} | {:>14} | {:>8}".format(
        "n", "log_norm", "||T_n||_F", "||J·v||/||v||", "time[s]"))
    print("-" * 64)

    for n in range(n_layers):
        t_layer0 = time.time()
        theta, logn = hotrg_step_flat(theta)
        theta.block_until_ready()
        t_layer1 = time.time()

        Tn = theta.reshape((chi, chi, chi, chi))
        fro = jnp.linalg.norm(Tn).item()
        cumulative_log_norm += float(logn)

        # Random test vector for Jacobian
        key, sk = random.split(key)
        v = random.normal(sk, theta.shape)
        v = v / jnp.linalg.norm(v)
        Jv = apply_J(theta, v)
        Jv_norm_ratio = (jnp.linalg.norm(Jv) / jnp.linalg.norm(v)).item()

        print("{:4d} | {:12.6f} | {:14.6e} | {:14.6e} | {:8.3f}".format(
            n, float(logn), fro, Jv_norm_ratio, t_layer1 - t_layer0))

    print("\nTotal accumulated log_norm (proxy for log Z):",
          f"{cumulative_log_norm:.6f}")
    print("Final tensor Frobenius norm:",
          float(jnp.linalg.norm(theta.reshape((chi, chi, chi, chi)))))

    return theta.reshape((chi, chi, chi, chi))


if __name__ == "__main__":
    # You probably want to start with something like D_cut=16 or 32
    # to make sure everything is happy before cranking to ~64.
    #
    # 20 layers with chi~64 will be heavy but A100-class hardware
    # should survive. Adjust n_layers, D_cut as needed.
    T_final = run_rg(beta=5.5, D_cut=32, n_layers=20, seed=123)

ModuleNotFoundError: No module named 'su3_irrep_tensor'